###**Step 1A - Create our support documents.**

In [ ]:
support_documents = [
    """
    Billing - Duplicate Charge

    If a customer reports being charged twice for the same transaction,
    verify the transaction history and payment records.

    If a duplicate charge is confirmed, the billing team should initiate
    a refund for the duplicate transaction.

    Refunds may take 5-7 business days to appear in the customer's account.
    """,

    """
    Billing - Failed Payment

    If a customer's payment fails, ask them to verify their payment
    information and ensure sufficient funds are available.

    The customer may retry the payment or use a different payment method.

    If the problem continues, the ticket should be routed to the
    Billing & Payments team.
    """,

    """
    Technical Support - Login Issue

    If a customer cannot log in, first ask them to verify their email
    address and password.

    The customer should try resetting their password using the
    Forgot Password option.

    If the issue continues after a password reset, the ticket should
    be escalated to Technical Support.
    """,

    """
    Technical Support - Website Outage

    If the production website is unavailable for multiple customers,
    treat the issue as a high-priority technical incident.

    The technical team should investigate service availability,
    server health, and recent deployments.

    Widespread outages should be escalated immediately.
    """,

    """
    Refund - Product Return

    Customers requesting a product return should provide their order
    number and reason for the return.

    Eligible products can be returned within 30 days of purchase.

    After the returned product is received and inspected, the refund
    will be processed to the original payment method.
    """,

    """
    Refund - Refund Status

    Customers asking about an existing refund should provide their
    order number or refund reference.

    Approved refunds normally take 5-7 business days to appear in
    the original payment method.

    If the refund has not appeared after this period, the case should
    be reviewed by the Returns & Exchanges team.
    """
]

In [ ]:
print("Number of support documents: ", len(support_documents))

print("\nExample document:")
print(support_documents[0])

Number of support documents:  6

Example document:

    Billing - Duplicate Charge

    If a customer reports being charged twice for the same transaction,
    verify the transaction history and payment records.

    If a duplicate charge is confirmed, the billing team should initiate
    a refund for the duplicate transaction.

    Refunds may take 5-7 business days to appear in the customer's account.
    


**We start with 6 documents rather than introducing PDFs, vector databases, APIs and LLMs all at once.**

###**Step 2: Understand and Create Embeddings.**

An embedding converts text into numbers that represent its meaning.

**Step 2A - Install Sentence Transformers**

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Step 2B: Converts our knowlede base into embeddings.**

In [ ]:
document_embeddings = embedding_model.encode(support_documents)

print("Number of documents: ", len(support_documents))
print("Embedding Shape: ", document_embeddings.shape)

Number of documents:  6
Embedding Shape:  (6, 384)


6 - Documents

384 - each documents has 384 numbers

In [ ]:
print(document_embeddings[0][:10])

[-0.06929141 -0.02217205  0.02863979 -0.02062539 -0.03196434 -0.06968139
  0.00642979 -0.07783861  0.09854957  0.02140895]


###**Step 3 - Retrieve the Most Relevant Document.**

**Step 3A : Create a test ticket**

In [ ]:
customer_ticket = """
I was charged twice for my subscription.
How can I get my money back?
"""

print(customer_ticket)


I was charged twice for my subscription.
How can I get my money back?



**Step 3B : Convert the ticket into an Embedding**

In [ ]:
ticket_embedding = embedding_model.encode([customer_ticket])
print("Ticket embedding shape: ", ticket_embedding.shape)

Ticket embedding shape:  (1, 384)


**Step 3C : Calculate Similarity**

In [ ]:
# we will use cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    ticket_embedding,
    document_embeddings
)[0]
print(similarities)

[0.58408797 0.45408165 0.31650126 0.1449934  0.34879673 0.28959116]


**Step 3D : Find the best document**

In [ ]:
best_match_index = similarities.argmax()

print("Best Document Index: ", best_match_index)
print("Similarituy Score: ", similarities[best_match_index])

print("\nRetrieved Document: ")
print(support_documents[best_match_index])

Best Document Index:  0
Similarituy Score:  0.58408797

Retrieved Document: 

    Billing - Duplicate Charge

    If a customer reports being charged twice for the same transaction,
    verify the transaction history and payment records.

    If a duplicate charge is confirmed, the billing team should initiate
    a refund for the duplicate transaction.

    Refunds may take 5-7 business days to appear in the customer's account.
    


**Analysis** - It worked, it correctly retrived the document.

###**Step 4 - Create a Retrieval Function**

In [ ]:
# instead of repeating all the embedding code every time, create a function

def retrieve_document(ticket_text):

  # Convert customer ticket into an embedding
  ticket_embedding = embedding_model.encode([ticket_text])

  # Compare the ticket with all the documents
  similarities = cosine_similarity(
      ticket_embedding,
      document_embeddings
  )[0]

  # Find the document with the highest similarity
  best_match_index = similarities.argmax()

  return {
      "document" : support_documents[best_match_index],
      "score": similarities[best_match_index],
      "index" : best_match_index
  }

In [ ]:
# Lets test the function

test_ticket = """
I cannot access my account.
I forgot my password and the login keeps failings.
"""

result = retrieve_document(test_ticket)

print("Document Index: ", result["index"])
print("Similarity Score: ", result["score"])
print("\nRetrieved Document: ")
print(result["document"])

Document Index:  2
Similarity Score:  0.6208638

Retrieved Document: 

    Technical Support - Login Issue

    If a customer cannot log in, first ask them to verify their email
    address and password.

    The customer should try resetting their password using the
    Forgot Password option.

    If the issue continues after a password reset, the ticket should
    be escalated to Technical Support.
    


**Analysis** - It worked our function is function is working perfectly.

###**Step 5 - Generation**

For the generation part, we need an LLM.

The important point is that the answer is generated using retrieved company knowledge, rather than asking the LLM to answer entirely from its own knowledge.

**Step 5A - Install Transformers**

In [ ]:
#Installs Hugging Face’s transformers (for models) and accelerate (for efficient hardware usage).
!pip install -q transformers accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 loaded successfully!


In [ ]:
"""  pipeline is a high-level API that makes it easy to run tasks like text generation,
translation, summarization, etc., without writing low-level model code. """

from transformers import pipeline

generator = pipeline(
    "text-generation",   # task type. It means the model takes text as input and produces text as output.
    model= "google/flan-t5-base" # loads the FLAN‑T5 base model, a fine‑tuned version of T5 designed for instruction following.
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel

###**Step 5B - Test the LLM by itself**

In [ ]:
# check if the generation works
prompt = """
You are a customer support agent.

Customer issue:
The customer cannot log in to their account.

Write a short and polite response telling the customer what they should do.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

outputs = generator_model.generate(
    **inputs,
    max_new_tokens=100
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response) # This is the response generation step in your support‑ticket pipeline.



I'm sorry to hear that. I'm trying to log in to your account.


###**Step 6 - Connect Retrieval + Generation**

In [ ]:
def generate_rag_response(ticket_text):

    retrieved = retrieve_document(ticket_text)
    context = retrieved["document"]

    prompt = f"""
Answer the customer using the support information.

SUPPORT INFORMATION:
{context}

CUSTOMER:
{ticket_text}

INSTRUCTIONS:
Tell the customer exactly what steps they should take.
Do not ask if they need anything else.
Do not invent information.

ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=120
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return {
        "response": response,
        "retrieved_document": context,
        "similarity_score": retrieved["score"]
    }

In [ ]:
customer_ticket = """
I forgot my password and now I cannot log in to my account.
What should I do?
"""

rag_result = generate_rag_response(customer_ticket)

print("Generated Response:")
print(rag_result["response"])

print("\nSimilarity Score:")
print(rag_result["similarity_score"])

print("\nRetrieved Knowledge:")
print(rag_result["retrieved_document"])

Generated Response:
if the customer cannot log in to their account.

Similarity Score:
0.5954299

Retrieved Knowledge:

    Technical Support - Login Issue

    If a customer cannot log in, first ask them to verify their email
    address and password.

    The customer should try resetting their password using the
    Forgot Password option.

    If the issue continues after a password reset, the ticket should
    be escalated to Technical Support.
    


**Analysis**- The response is not great, the retreival was wortking correctly but the response id not right.
We ill try with different model.

###**Step 8 - Load the new model.**

In [ ]:
"""Let's try Qwen2.5-1.5B-Instruct, a small instruction-following model that is more
appropriate for generating support responses than FLAN-T5-base."""

from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
    )

print("Qwen loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully!


###**Step 8 — Test Qwen generation**

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful customer support agent."
    },
    {
        "role": "user",
        "content": """
I forgot my password and cannot log in to my account.
Please write a short and polite customer support response.
"""
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(generator_model.device)

outputs = generator_model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False
)

# Remove the original prompt tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(response)

Hello! I'm sorry to hear that you're having trouble logging into your account. Can you please provide me with your email address so we can assist you further?


**Analysis:** - It worked properly, much better than FLAN-T5 output.
Lets now connect the model to retriever document.

###**Step 9 - Connect Qwen to your retriever.**

In [ ]:
def generate_rag_response(ticket_text):

    # 1. Retrieve relevant support document
    retrieved = retrieve_document(ticket_text)
    context = retrieved["document"]

    # 2. Create messages for Qwen
    messages = [
        {
            "role": "system",
            "content": (
                "You are a customer support agent. "
                "Answer the customer using ONLY the provided support information. "
                "Do not invent policies, steps, or information."
            )
        },
        {
            "role": "user",
            "content": f"""
SUPPORT INFORMATION:
{context}

CUSTOMER ISSUE:
{ticket_text}

Write a short, polite response directly to the customer.
"""
        }
    ]

    # 3. Apply Qwen chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 4. Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(generator_model.device)

    # 5. Generate
    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    # 6. Remove prompt tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    # 7. Decode response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return {
        "response": response,
        "retrieved_document": context,
        "similarity_score": retrieved["score"]
    }

In [ ]:
# Ticket 1
customer_ticket = """
I forgot my password and now I cannot log in to my account.
What should I do?
"""

rag_result = generate_rag_response(customer_ticket)

print("Generated Response:")
print(rag_result["response"])

print("\nSimilarity Score:")
print(rag_result["similarity_score"])

print("\nRetrieved Knowledge:")
print(rag_result["retrieved_document"])

Generated Response:
Hello! Thank you for reaching out with your login issue. To help resolve it quickly, could you please confirm if you have tried our Forgot Password feature? If that doesn't work, we recommend verifying your email address as well since this might also prevent any issues. Please let us know how we can assist further.

Similarity Score:
0.5954299

Retrieved Knowledge:

    Technical Support - Login Issue

    If a customer cannot log in, first ask them to verify their email
    address and password.

    The customer should try resetting their password using the
    Forgot Password option.

    If the issue continues after a password reset, the ticket should
    be escalated to Technical Support.
    


**Analysis** - It worked but the response doesn't say that the customer query will ve forwarded to Technical team.
We can make this modification in next steps.

In [ ]:
# Ticket 2:
customer_ticket = """
I was charged twice for my subscription this month.
I need the extra charge refunded.
"""

rag_result = generate_rag_response(customer_ticket)

print("Generated Response:")
print(rag_result["response"])

print("\nSimilarity Score:")
print(rag_result["similarity_score"])

print("\nRetrieved Knowledge:")
print(rag_result["retrieved_document"])

Generated Response:
Dear [Customer],

Thank you for bringing this issue to our attention. We have reviewed your transaction history and can confirm that there has been an additional charge for your subscription this month. 

To resolve this, we will process a refund for the duplicate charge. This refund should be credited to your account within approximately 5-7 business days. Please allow some time for the funds to transfer into your account.

If you have any further questions or concerns, please do not hesitate to contact us at [contact number/email].

We apologize for any inconvenience this may cause and appreciate your patience as we address this matter promptly.

Best regards,

[Your Name]  
Customer Support Team

Similarity Score:
0.5752772

Retrieved Knowledge:

    Billing - Duplicate Charge

    If a customer reports being charged twice for the same transaction,
    verify the transaction history and payment records.

    If a duplicate charge is confirmed, the billing team sh

**Analysis** : We got a strong result, and Qwen performed better, but we noticed that the knowledge base should be first verify it first, before refunding the money.

In [ ]:
# Let's test one more-

# Ticket 3:

customer_ticket = """
I returned my product last week and I want to know
when I will receive my refund.
"""

rag_result = generate_rag_response(customer_ticket)

print("Generated Response:")
print(rag_result["response"])

print("\nSimilarity Score:")
print(rag_result["similarity_score"])

print("\nRetrieved Knowledge:")
print(rag_result["retrieved_document"])

Generated Response:
Dear [Customer],
Thank you for returning your product promptly. We appreciate your understanding that it takes time to process returns. Typically, we aim to process refunds within 1-2 business days after receiving the item. Please keep an eye on your account for updates once we have completed the processing. If you need any further assistance, feel free to reach out.
Best regards,
[Your Name]  
Customer Support

Similarity Score:
0.697382

Retrieved Knowledge:

    Refund - Product Return

    Customers requesting a product return should provide their order
    number and reason for the return.

    Eligible products can be returned within 30 days of purchase.

    After the returned product is received and inspected, the refund
    will be processed to the original payment method.
    


**Analysis :**
The customer asked for status, but the retreiver selected product return. This is hallucination. This information does not exist in the retrieved document.

###**Step 10: Top-3 retrieval**

Right now your function only shows the single highest-scoring document, let's see what the retriever ranked second and third.

In [ ]:
def retrieve_top_documents(ticket_text, top_k=3):

    ticket_embedding = embedding_model.encode([ticket_text])

    similarities = cosine_similarity(
        ticket_embedding,
        document_embeddings
    )[0]

    # Sort documents from highest similarity to lowest
    top_indices = similarities.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "index": index,
            "score": similarities[index],
            "document": support_documents[index]
        })

    return results

In [ ]:
test_ticket = """
I returned my product last week and I want to know
when I will receive my refund.
"""

top_results = retrieve_top_documents(test_ticket)

for result in top_results:
    print("Document Index:", result["index"])
    print("Similarity Score:", result["score"])
    print(result["document"])
    print("-" * 50)

Document Index: 4
Similarity Score: 0.697382

    Refund - Product Return

    Customers requesting a product return should provide their order
    number and reason for the return.

    Eligible products can be returned within 30 days of purchase.

    After the returned product is received and inspected, the refund
    will be processed to the original payment method.
    
--------------------------------------------------
Document Index: 5
Similarity Score: 0.5859579

    Refund - Refund Status

    Customers asking about an existing refund should provide their
    order number or refund reference.

    Approved refunds normally take 5-7 business days to appear in
    the original payment method.

    If the refund has not appeared after this period, the case should
    be reviewed by the Returns & Exchanges team.
    
--------------------------------------------------
Document Index: 0
Similarity Score: 0.3429888

    Billing - Duplicate Charge

    If a customer reports being char

**Analysis** : We can see that the required output is at #2 number.

*   #1 Product Return       0.6974
*   #2 Refund Status        0.5860
*   #3 Duplicate Charge     0.3430

Maybe because the ticket has Product name in it hence its pickking the Product refunds instead of refund status.
We will try with top 2 now.

**Next Step : Give Qwen the Top-2 documents**

In [ ]:
def generate_rag_response_top2(ticket_text):

    # Retrieve top 2 documents
    retrieved_docs = retrieve_top_documents(
        ticket_text,
        top_k=2
    )

    context = "\n\n---\n\n".join(
        [result["document"] for result in retrieved_docs]
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a customer support agent. "
                "Answer using ONLY the provided support information. "
                "Do not invent policies, timelines, contact details, "
                "or actions that are not stated in the support information."
            )
        },
        {
            "role": "user",
            "content": f"""
SUPPORT INFORMATION:

{context}

CUSTOMER ISSUE:

{ticket_text}

Write a short and polite response directly to the customer.

Important:
- Use the information that best matches the customer's question.
- Do not claim that an action has already been completed.
- Do not invent any information.
"""
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(generator_model.device)

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return {
        "response": response,
        "retrieved_documents": retrieved_docs
    }

In [ ]:
customer_ticket = """
I returned my product last week and I want to know
when I will receive my refund.
"""

result = generate_rag_response_top2(customer_ticket)

print("Generated Response:")
print(result["response"])

Generated Response:
Dear [Customer],
Thank you for your inquiry regarding your refund status. Based on our policy, approved returns typically take 5-7 business days to process and appear in the original payment method. Please note that if the refund has not arrived after this timeframe, it may be necessary to review the case with the Returns & Exchanges team. We appreciate your patience as we work through this matter.
Best regards,
[Your Name]  
Customer Support Agent


###**Step 11 — Combine Phase 1 + RAG**

**Step 11A — Bring the Phase 1 files into this notebook**

In [ ]:
import os

print(os.listdir("/content"))

['.config', 'urgency_model.pkl', 'urgency_vectorizer.pkl', 'category_model.pkl', 'category_vectorizer.pkl', 'sample_data']


**Step 11B — Load the Phase 1 models**

In [ ]:
import joblib

# Load category model
category_vectorizer = joblib.load(
    "/content/category_vectorizer.pkl"
)

category_model = joblib.load(
    "/content/category_model.pkl"
)

# Load urgency model
urgency_vectorizer = joblib.load(
    "/content/urgency_vectorizer.pkl"
)

urgency_model = joblib.load(
    "/content/urgency_model.pkl"
)

print("Phase 1 models loaded successfully!")

Phase 1 models loaded successfully!


**Step 11C — Test the Phase 1 models**

In [ ]:
# Lets test before combining everything
def predict_ticket(ticket_text):

    # Category prediction
    category_features = category_vectorizer.transform([ticket_text])
    category = category_model.predict(category_features)[0]

    # Urgency prediction
    urgency_features = urgency_vectorizer.transform([ticket_text])
    urgency = urgency_model.predict(urgency_features)[0]

    return {
        "category": category,
        "urgency": urgency
    }


In [ ]:
test_ticket = """
Our production website is unavailable and customers
cannot log in. Please investigate urgently.
"""

prediction = predict_ticket(test_ticket)

print("Category:", prediction["category"])
print("Urgency:", prediction["urgency"])

Category: bug
Urgency: high


**Step 11D — Add Priority + Routing**

In [ ]:
priority_mapping = {
    "high": "P1",
    "medium": "P2",
    "low": "P3"
}

queue_mapping = {
    "billing": "Billing & Payments",
    "bug": "Technical Support",
    "refund": "Returns & Exchanges",
    "other": "General Support"
}

action_mapping = {
    "P1": "Escalate immediately and notify the technical team.",
    "P2": "Assign to the relevant support team for prompt review.",
    "P3": "Add to the normal support queue."
}

print("Business mappings loaded successfully!")

Business mappings loaded successfully!


**Step 11E — Create the ticket analysis function**

In [ ]:
def analyze_ticket(ticket_text):

    # Get ML predictions
    prediction = predict_ticket(ticket_text)

    category = prediction["category"]
    urgency = prediction["urgency"]

    # Apply business rules
    priority = priority_mapping.get(urgency, "P2")
    queue = queue_mapping.get(category, "General Support")
    action = action_mapping.get(
        priority,
        "Assign to the support team for review."
    )

    return {
        "category": category,
        "urgency": urgency,
        "priority": priority,
        "queue": queue,
        "action": action
    }

In [ ]:
test_ticket = """
Our production website is unavailable and customers
cannot log in. Please investigate urgently.
"""

analysis = analyze_ticket(test_ticket)

print("Category:", analysis["category"])
print("Urgency:", analysis["urgency"])
print("Priority:", analysis["priority"])
print("Queue:", analysis["queue"])
print("Recommended Action:", analysis["action"])

Category: bug
Urgency: high
Priority: P1
Queue: Technical Support
Recommended Action: Escalate immediately and notify the technical team.


**Step 11F — Combine ML + Business Logic + RAG**

In [ ]:
def support_copilot(ticket_text):


    # 1. ML Ticket Analysis

    analysis = analyze_ticket(ticket_text)


    # 2. RAG Response

    rag_result = generate_rag_response_top2(ticket_text)


    # 3. Combine Everything

    return {
        "category": analysis["category"],
        "urgency": analysis["urgency"],
        "priority": analysis["priority"],
        "queue": analysis["queue"],
        "action": analysis["action"],
        "response": rag_result["response"],
        "retrieved_documents": rag_result["retrieved_documents"]
    }

In [ ]:
# Test the system
ticket = """
Our production website is unavailable and multiple customers
cannot log in. Please investigate this urgently.
"""

result = support_copilot(ticket)

print("=== SUPPORT COPILOT RESULT ===")

print("\nCategory:", result["category"])
print("Urgency:", result["urgency"])
print("Priority:", result["priority"])
print("Queue:", result["queue"])
print("Recommended Action:", result["action"])

print("\nAI Suggested Response:")
print(result["response"])

print("\nRetrieved Knowledge:")

for i, doc in enumerate(result["retrieved_documents"], start=1):
    print(f"\nDocument {i}")
    print("Similarity Score:", doc["score"])
    print(doc["document"])

=== SUPPORT COPILOT RESULT ===

Category: bug
Urgency: high
Priority: P1
Queue: Technical Support
Recommended Action: Escalate immediately and notify the technical team.

AI Suggested Response:
Dear [Customer],
I apologize for the inconvenience caused by the current issues with our production website. We have initiated an investigation into the problem to ensure it does not affect other users. Our technical team will thoroughly examine server health, recent deployments, and service availability. Should you encounter further difficulties logging in, please follow these steps:

1. Verify your email address and password.
2. Try resetting your password using the "Forgot Password" option if prompted.

If the login issue persists beyond these steps, we recommend escalating the matter to our Technical Support team at your earliest convenience. Thank you for your patience and understanding during this time.

Best regards,
[Your Name]  
Customer Support Team

Retrieved Knowledge:

Document 1
Si

**Analysis** : This works fine.
But there is still little hallucination in phrases such as "We have initiated an investigation" the system know what to do, but an investigation has begun which the system does not know. We should get more ground base response.